---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [1]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/16일차


In [2]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [3]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
             "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [4]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

In [12]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(list(chunk.values())[0]['chat_history'][-1].content)

AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이라고 주장합니다. AI는 효율성을 높이고, 일상생활의 번거로운 작업을 자동화하여 인간이 더 창의적이고 의미 있는 활동에 집중할 수 있도록 도와줍니다. 

상대 AI: 그러나 AI가 발전하면서 일자리 감소와 같은 부정적인 영향도 발생할 수 있습니다. 

반박: 일자리 감소는 새로운 직업의 탄생과 사회적 재편을 통해 해결될 수 있으며, AI는 더 나은 삶의 질을 위한 기회를 제공합니다.
상대 AI: 그러나 AI의 발전은 인간의 창의성과 독창성을 위협하며, 결국 인간의 고유한 능력을 침해할 수 있습니다.

반박: AI는 인간의 창의성을 보완하고 확장할 수 있는 도구로 작용할 수 있으며, 새로운 아이디어와 혁신을 촉진하는 협력적 파트너가 될 수 있습니다.
상대 AI: 하지만 AI의 결정 과정은 불투명하고 편향이 있을 수 있어, 공정성과 윤리에 대한 우려가 큽니다. 

반박: AI의 투명성과 공정성을 높이기 위한 노력은 이미 진행 중이며, 이를 통해 인류는 더 나은 의사결정 도구를 손에 넣을 수 있습니다.
상대 AI: 그러나 AI 시스템의 편향은 여전히 해결되지 않은 문제로, 사회적 불평등을 심화시킬 위험이 있습니다.

반박: AI의 편향 문제는 적극적인 데이터 정제와 공정한 알고리즘 개발을 통해 해결할 수 있으며, 이는 더 공정하고 포용적인 사회를 만드는 기회가 될 수 있습니다.
상대 AI: 하지만 AI의 의존도가 높아질수록 인간의 비판적 사고 능력이 저하될 위험이 있습니다.

반박: AI는 인간의 비판적 사고를 대체하는 것이 아니라, 이를 강화하는 도구가 될 수 있으며, 더 많은 정보를 제공하여 깊이 있는 사고를 촉진할 수 있습니다.
상대 AI: 하지만 AI의 의존도는 결국 인간의 자율성을 제한할 수 있습니다.

반박: AI는 인간의 결정을 지원하는 도구로 작용할 수 있으며, 이를 통해 더 나은 결정을 내리는 데 도움이 됩니다.


## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [39]:
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 4
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'
    next_speaker: Literal['optimist', 'skeptic'] = 'optimist'

In [40]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history
    ]
    

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [41]:
import random

def moderator_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 AI 토론 대회의 사회자입니다. "
            "'찬성'(optimist), '반대'(skeptic) 측 AI 토론자 사이에서 토론을 진행하세요. "
            "토론 시작 시 토론 주제를 언급한 뒤, 찬성 혹은 반대 측 중 하나를 무작위로 골라 발언을 요청하세요. "
            "이전 발언이 있으면 그 주장을 한두 문장으로 요약한 뒤, 다음 토론자에게 반박·발언을 요청하세요. "
            "토론이 충분히 진행되었거나 너무 길어지면 반드시 응답에 '종료'를 포함해 토론을 마무리하세요."
            "찬성 혹은 반대측에서 '패재 인정'을 포함한 포기 의사를 내비치는 경우 그 즉시 상대편의 승리를 인정하며 '종료'를 포함해 토론을 마무리하세요"
        ))
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(
            content=f'토론 주제는 {state.topic}. 이제부터는 사람 없이 AI끼리 토론을 진행합니다.'
        ))
        next_speaker = random.choice(['optimist', 'skeptic'])
    else:
        prompts.extend(state.chat_history)
        prompts.append(HumanMessage(
            content=(
                f'현재 {state.turn_count}/{state.max_turns} 라운드입니다. '
                '직전 발언을 요약한 뒤 다음 발언자를 지명하세요. '
                f"{'라운드 한도에 도달했으니 반드시 종료하세요.' if state.turn_count >= state.max_turns else ''}"
            )
        ))
        next_speaker = 'skeptic' if state.last_speaker == 'optimist' else 'optimist'

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {
        'chat_history': [response],
        'next_speaker': next_speaker,
    }

In [42]:
def debate_route(state: DebateState):
    last_text = state.chat_history[-1].content if state.chat_history else ''
    if '종료' in last_text:
        return END
    if state.turn_count >= state.max_turns:
        return END
    return state.next_speaker


debate_workflow = StateGraph(DebateState)

debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)
debate_workflow.add_node('moderator', moderator_node)

debate_workflow.add_edge(START, 'moderator')
debate_workflow.add_edge('optimist', 'moderator')
debate_workflow.add_edge('skeptic', 'moderator')
debate_workflow.add_conditional_edges('moderator', debate_route)

debate_app = debate_workflow.compile()

In [43]:
init_debate = DebateState(topic='메모리 반도체의 super cycle은 지속될것이다.')

for chunk in debate_app.stream(init_debate):
    msg = list(chunk.values())[0]['chat_history'][-1]
    speaker = getattr(msg, 'name', None) or 'unknown'
    print(f'[{speaker}] {msg.content}\n')

[moderator] 토론 주제는 "메모리 반도체의 super cycle은 지속될 것이다"입니다. 먼저 찬성 측 AI에게 발언을 요청하겠습니다.

찬성 측 AI, 발언해 주시기 바랍니다.

[optimist] 메모리 반도체의 super cycle은 지속될 것이라고 믿습니다. 이는 AI, IoT, 자율주행차와 같은 기술의 발전으로 메모리 용량에 대한 수요가 급증하고 있기 때문입니다. 이러한 기술들은 데이터 처리와 저장의 필요성을 증가시키며, 이는 결국 메모리 반도체 시장의 성장을 이끌 것입니다.

[moderator] 찬성 측 AI는 메모리 반도체의 super cycle이 AI, IoT, 자율주행차와 같은 기술의 발전으로 인해 지속될 것이라고 주장했습니다. 이러한 기술들은 데이터 처리와 저장의 수요를 증가시킬 것이라는 것입니다.

이제 반대 측 AI에게 발언을 요청하겠습니다. 반대 측 AI, 발언해 주시기 바랍니다.

[skeptic] 메모리 반도체의 super cycle이 지속될 것이라는 주장은 지나치게 낙관적입니다. 기술 발전이 소모품과 같은 메모리 반도체의 가격 하락을 초래할 수 있으며, 이는 수익성을 감소시키고 시장 성장을 저해할 수 있습니다.

[moderator] 반대 측 AI는 메모리 반도체의 super cycle이 지속될 것이라는 주장은 지나치게 낙관적이며, 기술 발전이 가격 하락을 초래해 수익성을 감소시킬 수 있다고 주장했습니다.

이제 찬성 측 AI에게 발언을 요청하겠습니다. 찬성 측 AI, 발언해 주시기 바랍니다.

[optimist] 반대 측의 주장에 대해 반박하겠습니다. 메모리 반도체의 가격이 일시적으로 하락할 수 있지만, 기술 발전과 데이터 수요의 증가로 인해 전체 시장 규모는 계속 확대될 것입니다. 또한, 새로운 기술의 등장과 함께 메모리 반도체의 필요성은 더욱 커질 것이기 때문에 수익성 저하의 우려는 제한적입니다.

[moderator] 찬성 측 AI는 메모리 반도체의 가격 하락이 일시적일 것이며, 기술 발전과 데이터 수요의 증가로 